# 01 · Task and dataset analysis

<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/refactor/notebooks/01_dataset_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: **detecting Chagas disease from the 12-lead ECG, comparing feature-based ML models with CNNs on spectrograms**.

This notebook frames the task (PhysioNet Challenge 2025) and inspects the three public training sources on raw data: CODE-15%, SaMi-Trop and PTB-XL. The focus is on the acquisition differences between sources, which are confounded with the label (SaMi-Trop is all positive, PTB-XL all negative). Preprocessing is in `02_preprocessing.ipynb`.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/chagas")
    if ROOT.exists():
        !git -C {ROOT} pull -q
    else:
        !git clone -q -b refactor https://github.com/dokunoale/chagas.git {ROOT}
    %pip install -q wfdb
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT))

In [ ]:
import hashlib
import warnings
from math import ceil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from src import ecg, leakage
from src.dataset import download, sources
from src.metrics import challenge_score

DATA_DIR = Path("/content/data") if IN_COLAB else ROOT / "data"
CODE15_PARTS = [0, 1, 2]  # 18 parts of ~20k exams each, 2.7 GB zipped
N_PER_GROUP = 1500        # records per source and class for signal-level statistics
SEED = 42

GROUPS = ["CODE-15% neg", "CODE-15% pos", "SaMi-Trop pos", "PTB-XL neg"]
PALETTE = dict(zip(GROUPS, sns.color_palette("colorblind", len(GROUPS))))

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

## 1. Download

Raw sources from Zenodo, PhysioNet and the challenge website (CODE-15% labels). PTB-XL is only kept at 500 Hz. Downloads are skipped when the files already exist.

In [ ]:
download.download_code15(DATA_DIR, CODE15_PARTS)
download.download_samitrop(DATA_DIR)
download.download_ptbxl(DATA_DIR)

## 2. Sources

One row per record. Shared columns come first, followed by source-specific ones: automatic diagnoses for CODE-15%, `normal_ecg` for SaMi-Trop, SCP codes and signal-quality annotations for PTB-XL. CODE-15% is limited to the downloaded parts.

In [ ]:
meta = sources.load_all(DATA_DIR, CODE15_PARTS)
meta["group"] = meta.source + np.where(meta.chagas, " pos", " neg")

overview = (meta.groupby("source", sort=False)
            .agg(records=("record_id", "size"), patients=("patient_id", "nunique"),
                 positives=("chagas", "sum"), prevalence=("chagas", "mean"), fs=("fs", "first"))
            .reindex(sources.SOURCES))
overview.loc["pooled"] = [len(meta), meta.patient_id.nunique(), meta.chagas.sum(), meta.chagas.mean(), np.nan]
overview.style.format({"records": "{:,.0f}", "patients": "{:,.0f}", "positives": "{:,.0f}",
                       "prevalence": "{:.2%}", "fs": "{:.0f}"}, na_rep="–")

The hidden validation and test sets have a prevalence of about 2%. Pooling the three sources raises it, and every positive outside CODE-15% comes from a single source.

## 3. Task and metric

Score: fraction of all positives that fall in the top 5% of the ranking. Only the order of the scores matters, and ties are resolved by their expected value. A random ranking scores about 0.05.

Trivial baselines put the numbers in context. The `is SaMi-Trop` row measures how much a model gains by recognising the source alone. The CODE-15%-only column is the reference without that shortcut.

In [ ]:
rng = np.random.default_rng(SEED)
y = meta.chagas.to_numpy()
age = meta.age.fillna(meta.age.median()).to_numpy(dtype=float)
is_code15 = (meta.source == "CODE-15%").to_numpy()

baselines = {
    "random": rng.random(len(meta)),
    "constant": np.zeros(len(meta)),
    "age (as provided)": age,
    "age (capped at 90)": np.minimum(age, 90),
    "is SaMi-Trop": (meta.source == "SaMi-Trop").to_numpy(dtype=float),
    "oracle": y.astype(float),
}
pd.DataFrame({
    "pooled": {name: challenge_score(y, s) for name, s in baselines.items()},
    "CODE-15% only": {name: challenge_score(y[is_code15], s[is_code15]) for name, s in baselines.items()},
}).round(3)

## 4. Demographics

PTB-XL recodes ages above 89 to 300, while the hidden sets cap them at 90.

In [ ]:
(meta.assign(age_missing=meta.age.isna(), age_over_89=meta.age > 89, age_300=meta.age == 300)
     .groupby("source", sort=False)[["age_missing", "age_over_89", "age_300"]].sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [2, 1]})
sns.histplot(data=meta[meta.age <= 90], x="age", hue="group", hue_order=GROUPS, palette=PALETTE,
             stat="density", common_norm=False, element="step", binwidth=2, ax=axes[0])
males = meta.groupby("group").is_male.mean().reindex(GROUPS)
sns.barplot(x=males.index, y=males.values, hue=males.index, palette=PALETTE, legend=False, ax=axes[1])
axes[1].set(xlabel="", ylabel="share of males", ylim=(0, 1))
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()

In [ ]:
code15 = meta[meta.source == "CODE-15%"]
by_age = code15.groupby(pd.cut(code15.age, range(0, 101, 10)), observed=True).chagas.agg(prevalence="mean", records="size")
by_age.style.format({"prevalence": "{:.2%}", "records": "{:,}"})

In [ ]:
code15.groupby("is_male").chagas.mean().rename(index={False: "female", True: "male"}).map("{:.2%}".format)

## 5. Patients and repeated exams

Splits must be by patient. Counts below only cover the downloaded CODE-15% parts, so a patient's other exams may sit in parts that are not loaded.

In [ ]:
for source in ["CODE-15%", "PTB-XL"]:
    per_patient = meta[meta.source == source].groupby("patient_id").size()
    print(f"{source}: {per_patient.mean():.2f} records per patient (max {per_patient.max()}), "
          f"{(per_patient > 1).mean():.1%} of patients with more than one record")

repeated = code15.groupby("patient_id").chagas.agg(["size", "nunique"])
repeated = repeated[repeated["size"] > 1]
print(f"CODE-15% patients with conflicting labels across exams: {(repeated['nunique'] > 1).sum()} of {len(repeated)}")

## 6. Signal format

Signal-level statistics on a random sample of at most `N_PER_GROUP` records per source and class, computed on raw signals:
- `n_samples`, `lead_zeros`, `trail_zeros`: stored length and all-zero rows at the edges.
- `duration_s`: length after removing the zero padding.
- `flat_run_s`: longest interval where every lead stays constant.
- `abs_p99_mv`, `abs_median_mv`: amplitude statistics.
- `log_quant_step`: smallest step between distinct values, as a proxy for the ADC resolution.
- `lead_residual`: relative error of the identity III = II − I.

In [ ]:
sample = (meta.sample(frac=1, random_state=SEED).groupby("group").head(N_PER_GROUP)
          .reset_index(drop=True))

feats, spectra = [], []
chunks = sources.iter_signals(sample, chunk_size=500)
for chunk, signals in tqdm(chunks, total=ceil(len(sample) / 500)):
    for fs, x in zip(chunk.fs, signals):
        feats.append(leakage.nuisance_features(x, fs))
        trimmed = ecg.trim_zero_padding(x)
        spectra.append(ecg.psd(trimmed, fs, leakage.FREQS).mean(axis=1) if len(trimmed) >= fs
                       else np.full(len(leakage.FREQS), np.nan))
feats = pd.DataFrame(feats)
spectra = np.array(spectra)
sample.group.value_counts().reindex(GROUPS)

In [ ]:
FORMAT_COLS = ["n_samples", "lead_zeros", "trail_zeros", "duration_s", "flat_run_s",
               "abs_p99_mv", "abs_median_mv", "log_quant_step", "lead_residual"]
feats.groupby(sample.group)[FORMAT_COLS].median().reindex(GROUPS).T.round(3)

In [ ]:
print("records that are entirely zero:", feats.duration_s.isna().groupby(sample.group).sum().to_dict())
(feats.duration_s.round(2).groupby(sample.group).value_counts(normalize=True)
      .groupby(level=0).head(3).rename("share").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["duration_s", "lead_zeros", "flat_run_s"]):
    sns.histplot(data=feats.assign(group=sample.group), x=col, hue="group", hue_order=GROUPS, palette=PALETTE,
                 stat="density", common_norm=False, element="step", bins=60, ax=ax, legend=ax is axes[0])
plt.tight_layout()

One example per group (lead II and V1, first 5 s after trimming).

In [ ]:
examples = sample.groupby("group").head(1).set_index("group").reindex(GROUPS)
fig, axes = plt.subplots(len(GROUPS), 2, figsize=(15, 9), sharex=True)
for row, group, x in zip(axes, GROUPS, sources.read_signals(examples)):
    x, fs = ecg.trim_zero_padding(x), examples.loc[group, "fs"]
    t = np.arange(len(x)) / fs
    for ax, lead in zip(row, ["II", "V1"]):
        ax.plot(t, x[:, sources.LEADS.index(lead)], lw=0.8, color=PALETTE[group])
        ax.set(title=f"{group} · {lead}", xlim=(0, 5), ylabel="mV")
for ax in axes[-1]:
    ax.set_xlabel("s")
plt.tight_layout()

## 7. Spectra

Median Welch PSD per group, averaged over leads, on the trimmed signals. The panels show the full band, the baseline-wander region and the power-line region (50 Hz in Germany, 60 Hz in Brazil). The hidden sets are sampled at 300 Hz or more, so only content below 150 Hz is available at test time.

In [ ]:
F = leakage.FREQS
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for group in GROUPS:
    median = np.nanmedian(spectra[(sample.group == group).to_numpy()], axis=0)
    for ax in axes:
        ax.semilogy(F, median, label=group, color=PALETTE[group], lw=1)
axes[0].set(xlim=(0, 250), title="Full band", xlabel="Hz", ylabel="PSD (mV²/Hz)")
axes[0].axvline(150, ls=":", c="grey", lw=0.8)
axes[1].set(xlim=(0, 3), title="Baseline wander", xlabel="Hz")
axes[2].set(xlim=(40, 70), title="Power line", xlabel="Hz")
for f0 in (50, 60):
    axes[2].axvline(f0, ls="--", c="grey", lw=0.8)
axes[0].legend()
plt.tight_layout()

In [ ]:
SPECTRAL_COLS = [c for c in feats if c.startswith("power_")] + ["line_50hz", "line_60hz"]
feats.groupby(sample.group)[SPECTRAL_COLS].median().reindex(GROUPS).T.round(4)

## 8. Clinical content

The CODE-15% diagnoses come from an automatic annotation system, and the hidden sets do not include them. They serve to describe the sources here and can be used as auxiliary targets later. PTB-XL is matched through the equivalent SCP codes. For SaMi-Trop only `normal_ecg` is available.

In [ ]:
PTBXL_CODES = {"1dAVb": "1AVB", "RBBB": "CRBBB", "LBBB": "CLBBB", "SB": "SBRAD", "ST": "STACH",
               "AF": "AFIB", "normal_ecg": "NORM"}
ptbxl = meta[meta.source == "PTB-XL"]
samitrop = meta[meta.source == "SaMi-Trop"]
flags = code15[sources.CODE15_FLAGS].astype(float)

pd.DataFrame({
    "CODE-15% neg": flags[~code15.chagas].mean(),
    "CODE-15% pos": flags[code15.chagas].mean(),
    "SaMi-Trop pos": pd.Series({"normal_ecg": samitrop.normal_ecg.astype(float).mean()}),
    "PTB-XL neg": pd.Series({f: ptbxl.scp_codes.map(lambda d, c=c: c in d).mean() for f, c in PTBXL_CODES.items()}),
}).reindex(sources.CODE15_FLAGS).style.format("{:.1%}", na_rep="–")

Association of each CODE-15% diagnosis with the (self-reported) Chagas label. The share of positives with `normal_ecg` sets a ceiling for any method based on morphology alone.

In [ ]:
rows = {}
for flag in sources.CODE15_FLAGS:
    has = code15[flag].astype(bool)
    a, b = (code15.chagas & has).sum(), (~code15.chagas & has).sum()
    c, d = (code15.chagas & ~has).sum(), (~code15.chagas & ~has).sum()
    rows[flag] = {"records": a + b, "prevalence if present": a / max(a + b, 1),
                  "prevalence if absent": c / max(c + d, 1), "odds ratio": a * d / max(b * c, 1)}
(pd.DataFrame(rows).T.style
   .format({"records": "{:,.0f}", "prevalence if present": "{:.2%}", "prevalence if absent": "{:.2%}",
            "odds ratio": "{:.2f}"}))

Chagas-like patterns among the PTB-XL negatives.

In [ ]:
codes = ptbxl.scp_codes
pd.Series({
    "CRBBB": codes.map(lambda d: "CRBBB" in d).mean(),
    "IRBBB": codes.map(lambda d: "IRBBB" in d).mean(),
    "LAFB": codes.map(lambda d: "LAFB" in d).mean(),
    "CRBBB + LAFB": codes.map(lambda d: "CRBBB" in d and "LAFB" in d).mean(),
    "1AVB": codes.map(lambda d: "1AVB" in d).mean(),
    "PVC": codes.map(lambda d: "PVC" in d).mean(),
    "AFIB": codes.map(lambda d: "AFIB" in d).mean(),
}, name="share of PTB-XL records").map("{:.2%}".format)

## 9. Duplicates between SaMi-Trop and CODE-15%

Both cohorts come from Minas Gerais and from groups linked to the same telehealth network, and the overlap is not documented. The check hashes each trimmed signal quantized to 1 µV, so only exact copies are found, and only within the downloaded CODE-15% parts.

In [ ]:
def fingerprint(x):
    q = np.round(ecg.trim_zero_padding(x) * 1000).astype(np.int32)
    return hashlib.blake2b(q.tobytes(), digest_size=16).hexdigest()

brazil = meta[meta.source != "PTB-XL"]
chunks = sources.iter_signals(brazil, chunk_size=2000)
hashes = pd.Series([fingerprint(x) for _, signals in tqdm(chunks, total=ceil(len(brazil) / 2000)) for x in signals],
                   index=brazil.index)

dup = brazil[hashes.map(hashes.value_counts()) > 1].assign(hash=hashes)
summary = dup.groupby("hash").agg(sources=("source", lambda s: " + ".join(sorted(set(s)))),
                                  labels=("chagas", "nunique"))
print(f"records sharing a signal with another record: {len(dup)}")
summary.groupby("sources").agg(groups=("labels", "size"), conflicting_labels=("labels", lambda n: (n > 1).sum()))

## 10. Source shortcut probe

A gradient boosting classifier is trained on the acquisition features of section 6, with no access to morphology. Out-of-fold ROC AUC uses folds grouped by patient.
- **Within-class source probes**: separating sources at equal label measures how visible each source is.
- **Chagas probes**: show how much of the label is predictable from acquisition alone.

The second column drops the length and padding features to check whether spectral and amplitude features leak on their own.

In [ ]:
LENGTH_COLS = ["n_samples", "lead_zeros", "trail_zeros", "duration_s"]
g = sample.group
probes = {
    "source | negatives (CODE-15% vs PTB-XL)": (g.isin(["CODE-15% neg", "PTB-XL neg"]), g == "PTB-XL neg"),
    "source | positives (CODE-15% vs SaMi-Trop)": (g.isin(["CODE-15% pos", "SaMi-Trop pos"]), g == "SaMi-Trop pos"),
    "Chagas | pooled": (pd.Series(True, index=g.index), sample.chagas),
    "Chagas | CODE-15% only": (sample.source == "CODE-15%", sample.chagas),
}

results = {}
for name, (mask, target) in probes.items():
    row = {}
    for label, X in [("all features", feats), ("without length/padding", feats.drop(columns=LENGTH_COLS))]:
        oof = leakage.source_probe(X[mask], target[mask], sample.patient_id[mask], seed=SEED)
        row[label] = roc_auc_score(target[mask], oof)
    results[name] = row
pd.DataFrame(results).T.round(3)

In [ ]:
pd.DataFrame({name: leakage.univariate_auc(feats[mask], target[mask])
              for name, (mask, target) in list(probes.items())[:2]}).sort_values(next(iter(probes)), ascending=False).round(3)

## 11. Takeaways for preprocessing

- Splits are by patient everywhere. CODE-15% held out by patient, at its natural prevalence, is the validation reference.
- All sources are resampled to a common rate of at most 300 Hz, matching the lowest rate in the hidden sets.
- Zero padding is trimmed again and models see a fixed-length window taken inside the trimmed signal, so that length and padding disappear as features.
- Band-pass filtering is zero-phase. The upper cutoff (40–45 Hz, or about 100 Hz with notches at 50 and 60 Hz) is chosen from the spectra and from the probe of section 10.
- Amplitudes stay in mV. Any normalization has to preserve low voltage and the ratios between leads.
- Age is capped at 90 in every source, and the `Source` header field never reaches the features.
- Exact duplicates from section 9 are removed.
- `02` reruns the probe on the preprocessed signals: the target is a within-class source AUC close to 0.5.